# Notebook 01 — Data Understanding

**Milestone:** Phase 4 — Exploratory Data Analysis (Section 10.5)
**Source:** validated processed dataset `data/processed/airfoil_self_noise.csv`

Purpose: confirm the validated dataset loads correctly, inspect its dimensions and data types, produce descriptive statistics, verify data integrity, and document the role of each of the six canonical variables.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until the repository root (pyproject.toml + src/airfoil_noise) is found."""
    cur = start.resolve()
    for _ in range(6):
        if (cur / "pyproject.toml").exists() and (cur / "src" / "airfoil_noise").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Repository root could not be located")


REPO_ROOT = find_repo_root(Path.cwd())
PROCESSED_PATH = REPO_ROOT / "data" / "processed" / "airfoil_self_noise.csv"

print("Reading processed dataset:", PROCESSED_PATH)
df = pd.read_csv(PROCESSED_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Reading processed dataset: D:\Project Aero\airfoil-noise-prediction\data\processed\airfoil_self_noise.csv
Loaded 1,503 rows x 6 columns


,frequency_hz,angle_of_attack_deg,chord_length_m,free_stream_velocity_m_s,suction_side_displacement_thickness_m,scaled_sound_pressure_level_db
0,800,0.0,0.3048,71.3,0.002663,126.201
1,1000,0.0,0.3048,71.3,0.002663,125.201
2,1250,0.0,0.3048,71.3,0.002663,125.951
3,1600,0.0,0.3048,71.3,0.002663,127.591
4,2000,0.0,0.3048,71.3,0.002663,127.461


## Dataset dimensions

Report the number of observations and variables, and show a preview of the data.

In [2]:
print("Dimensions (rows, columns):", df.shape)
print("Number of rows:", len(df))
print("Number of columns:", df.shape[1])
print("\nFirst 10 rows:")
df.head(10)


Dimensions (rows, columns): (1503, 6)
Number of rows: 1503
Number of columns: 6

First 10 rows:


,frequency_hz,angle_of_attack_deg,chord_length_m,free_stream_velocity_m_s,suction_side_displacement_thickness_m,scaled_sound_pressure_level_db
0,800,0.0,0.3048,71.3,0.002663,126.201
1,1000,0.0,0.3048,71.3,0.002663,125.201
2,1250,0.0,0.3048,71.3,0.002663,125.951
3,1600,0.0,0.3048,71.3,0.002663,127.591
4,2000,0.0,0.3048,71.3,0.002663,127.461
5,2500,0.0,0.3048,71.3,0.002663,125.571
6,3150,0.0,0.3048,71.3,0.002663,125.201
7,4000,0.0,0.3048,71.3,0.002663,123.061
8,5000,0.0,0.3048,71.3,0.002663,121.301
9,6300,0.0,0.3048,71.3,0.002663,119.541


## Column data types

Confirm every column is numeric and report non-null counts per column.

In [3]:
dtypes = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dt) for dt in df.dtypes],
    "non_null_count": df.notna().sum(),
})
print(dtypes.to_string(index=False))


                               column   dtype  non_null_count
                         frequency_hz   int64            1503
                  angle_of_attack_deg float64            1503
                       chord_length_m float64            1503
             free_stream_velocity_m_s float64            1503
suction_side_displacement_thickness_m float64            1503
       scaled_sound_pressure_level_db float64            1503


## Descriptive statistics

Summary statistics (count, mean, std, min, quartiles, max) for every column.

In [4]:
print(df.describe().to_string())


       frequency_hz  angle_of_attack_deg  chord_length_m  free_stream_velocity_m_s  suction_side_displacement_thickness_m  scaled_sound_pressure_level_db
count   1503.000000          1503.000000     1503.000000               1503.000000                            1503.000000                     1503.000000
mean    2886.380572             6.782302        0.136548                 50.860745                               0.011140                      124.835943
std     3152.573137             5.918128        0.093541                 15.572784                               0.013150                        6.898657
min      200.000000             0.000000        0.025400                 31.700000                               0.000401                      103.380000
25%      800.000000             2.000000        0.050800                 39.600000                               0.002535                      120.191000
50%     1600.000000             5.400000        0.101600                 39.

## Data integrity

Check for missing values, exact duplicate rows, and non-finite values across the frame. The validated dataset should contain zero missing values, zero duplicates, and only finite numbers.

In [5]:
missing_total = int(df.isna().sum().sum())
duplicates = int(df.duplicated().sum())
arr = df.to_numpy(dtype=float)
non_finite = int((~np.isfinite(arr)).sum())

print("Missing values total:", missing_total)
print("Exact duplicate rows:", duplicates)
print("Non-finite values:", non_finite)
print("\nIntegrity OK:", bool(missing_total == 0 and duplicates == 0 and non_finite == 0))

Missing values total: 0
Exact duplicate rows: 0
Non-finite values: 0

Integrity OK: True


## Variable dictionary — all six canonical variables

| Column | Unit | Role |
|---|---|---|
| `frequency_hz` | Hz | Input feature — broadband frequency of the measured noise |
| `angle_of_attack_deg` | degrees | Input feature — airfoil angle of attack |
| `chord_length_m` | m | Input feature — airfoil chord length |
| `free_stream_velocity_m_s` | m/s | Input feature — free-stream velocity |
| `suction_side_displacement_thickness_m` | m | Input feature — suction-side displacement thickness |
| `scaled_sound_pressure_level_db` | dB | Target variable — scaled sound pressure level to predict |

The first five columns are the experimental input measurements; the last column is the regression target. The original raw file `airfoil_self_noise.dat` is preserved unchanged and was not loaded or modified in this notebook.